[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SaronJenish/zawdwadw/blob/main/Experiment_25_Restricted_Boltzmann_Machine_Energy_and_Reconstruction.ipynb)


# Experiment 25: Restricted Boltzmann Machine Energy and Reconstruction

CSA47 – Deep Learning. Run the setup cell first, then execute the experiment cells in order.


In [ ]:
# Common setup – run this cell first
import os, time, warnings, math
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, mean_absolute_error, mean_squared_error)
np.random.seed(42)
plt.rcParams["figure.figsize"] = (7, 4)
print("Setup complete")

## Experiment 25 – Restricted Boltzmann Machine Energy and Reconstruction

In [ ]:
# Binary RBM implemented with NumPy; small synthetic binary user-item matrix
rng=np.random.default_rng(25); V=(rng.random((300,20))>.6).astype(float)
nv,nh=V.shape[1],12; W=rng.normal(0,.01,(nv,nh)); a=np.zeros(nv); b=np.zeros(nh)
sig=lambda x:1/(1+np.exp(-np.clip(x,-30,30)))
def sample_prob(p): return (rng.random(p.shape)<p).astype(float)
def energy(v,h): return -v@a-h@b-np.sum((v@W)*h,axis=1)
errors=[]; free_energy=[]
for epoch in range(30):
    for start in range(0,len(V),30):
        v0=V[start:start+30]; ph0=sig(v0@W+b); h0=sample_prob(ph0)
        pv1=sig(h0@W.T+a); v1=sample_prob(pv1); ph1=sig(v1@W+b)
        W+=.05*(v0.T@ph0-v1.T@ph1)/len(v0); a+=.05*np.mean(v0-v1,axis=0); b+=.05*np.mean(ph0-ph1,axis=0)
    rec=sig(sig(V@W+b)@W.T+a)
    errors.append(np.mean((V-rec)**2))
    free_energy.append(np.mean(-np.sum(np.logaddexp(0,V@W+b),axis=1)-V@a))
print("Final reconstruction error:",errors[-1])
plt.plot(errors,label="Reconstruction MSE"); plt.plot(free_energy,label="Free energy"); plt.legend(); plt.show()
fig,axs=plt.subplots(2,5,figsize=(10,4))
for i,ax in enumerate(axs.flat):
    ax.imshow(np.vstack([V[i],rec[i]]),aspect="auto",cmap="gray"); ax.set_title("Original / recon"); ax.axis("off")
plt.tight_layout(); plt.show()